# Trader Execution

Walk-forward is the test design: train on the past, test on the next unseen stretch, with an
hard fenced split so nothing leaks across. This chapter's notebook runs the machine-learning 
workflow on Binance's 1-hour full-market data for all active USDT spot pairs (n=433) trading 
with three pinned currencies. This point-in-time screening method was adopted as recommended 
data standards for building cryptocurrency training data and machine learning pipelines. 

This was planned across five stages: i) building and processing the labelled 1h dataset, ii) running variable 
selection tests to confirm best subset of indicator features, iii) fitting and training models 
head-to-head between logistic regression, random forest, LightGBM meeting a 60/40 confidence filter, 
iv) testing models and tuning hyperparameters on blind data to nominate `best model`.

All model performance metrics are compiled in the `outputs/AA-evals/` folder, comparing features and 
the ATR-scaled label from `inputs/build_dataset_1h.py`, the final-year split from
`inputs/train_model_1h.py`, the estimator list and per-model scoring from `inputs/train_model.py`,
and the four-metric evaluation from `inputs/eval_report.py`. Run it from
the project `.venv`, which holds `pandas-ta` and `TA-Lib` lirbaries. 

### Environment

This notebook must run on the project .venv kernel: "Python (swing-trader .venv)", which holds joblib, scikit-learn, lightgbm, ccxt, pandas-ta and TA-Lib. In Jupyter you do NOT "activate" a venv in a shell, 
you select its kernel, Kernel menu >> Change Kernel >> "Python (swing-trader .venv)". If that kernel is not listed, register it once from a terminal, then reopen the notebook:

```
   /Volumes/PortableSSD/Github/day-trader/.venv/bin/python -m ipykernel install --user \
       --name day-trader --display-name "Python (day-trader .venv)"
```

In [1]:
import os, sys
from pathlib import Path


if sys.version_info[:2] < (3, 11):
    raise RuntimeError(f"Use the project .venv (Python 3.12); kernel is {sys.version.split()[0]}.")
try:
    import numpy as np, pandas as pd, joblib
    import matplotlib.pyplot as plt
except ModuleNotFoundError as e:
    raise RuntimeError(
        f"Missing '{e.name}'. This kernel is not the project .venv (running {sys.executable}). "
        "Switch to 'Python (day-trader .venv)' via Kernel -> Change Kernel, then re-run."
    ) from e

# Shared modeling + evaluation code: the SAME functions the scripts run.
for cand in ["inputs", os.path.join("..", "inputs")]:
    if os.path.isdir(cand):
        sys.path.insert(0, os.path.abspath(cand)); break
import build_dataset_1h as bd        # 1h features, ATR-scaled label, point-in-time screen
import train_model as tm             # shared: build_models, evaluate, confidence_filtered, costs
import train_model_1h as t1          # 1h load + final-year split
import eval_report
HAVE_LGBM = tm.HAVE_LGBM

def _layer(mod):
    try:
        __import__(mod); return "yes"
    except Exception:
        return "no"

OUTPUTS = Path("outputs"); MODEL_DIR = OUTPUTS / "3B-model-training"; MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"env ready  .  kernel: {sys.executable}")
print(f"python {sys.version.split()[0]}  .  lightgbm {'yes' if HAVE_LGBM else 'no'}"
      f"  .  pandas-ta {_layer('pandas_ta')}  .  TA-Lib {_layer('talib')}")

# --- inline table rendering (Positron-safe) -------------------------------------------
# Positron routes a bare display(df) to its live Data Explorer pane, which shows blank
# inline and dies on reload. show_table() emits a static HTML table that renders in the
# cell and survives reload, in Positron, Jupyter and VS Code alike.
from IPython.display import HTML, display, Markdown  # noqa: F401  (Markdown used later)

def show_table(x):
    obj = x.to_frame() if (hasattr(x, "to_frame") and getattr(x, "ndim", 2) == 1) else x
    display(HTML(obj.to_html()))

env ready  .  kernel: /Volumes/PortableSSD/Github/day-trader/.venv/bin/python


python 3.12.13  .  lightgbm yes  .  pandas-ta yes  .  TA-Lib yes


## Data Preparation

Everything between the raw exchange archives and the first model lives here: what the data is, how it was cleaned and screened, how it is split for honest testing, and how that split is audited before any score is trusted. The operations are defined once in `inputs/build_dataset_1h.py` and `inputs/train_model_1h.py`; the cells in this section load the prepared data, list the predictor families, and demonstrate the cleaning and the split audit.

### The dataset and label

One row per coin per hour, across the full active USDT spot market, each row kept only if that coin was actually tradable at that hour (the point-in-time screen, below). The predictors are scale-invariant (ratios and distances, so one model fits a $60k coin and a $0.20 coin). The outcome label is an ATR-scaled triple barrier on a short day-trade horizon: a win is a +2 ATR rise before a -1 ATR drop, within 48 hours.

### Cleaning and screening (in `build_dataset_1h.py`)

This work happens once in the build script, on demand, never on a schedule. It turns Binance's raw hourly archives and the trade-flow table into the clean dataset the notebook loads.

- Gap gate - hourly bars should arrive one per hour; a coin is dropped if more than 2% of hours are missing or any single blackout exceeds 3 days, because the indicators assume an unbroken timeline.
- Point-in-time screen - each (coin, hour) row is kept only if that coin would have been worth trading at that moment: enough daily turnover, acceptable volatility, sufficient history, and an acceptable spread (a Corwin-Schultz proxy). This also removes the survivorship-bias trap.
- Warmup and missing labels - some indicators need months of prior bars and the label needs future bars to resolve; the first rows of each coin, where neither can be computed, are dropped so every remaining row is complete.
- Timestamp normalisation - Binance switched from milliseconds to microseconds mid-2025, so one coin's files can mix both; the loader detects each value's unit and converts to a single timestamp.
- Parquet - the dataset is stored column-at-a-time as compressed binary with dtypes embedded, which speeds parsing and avoids the timestamp bugs CSV caused. Reading it manually needs `pandas` and `pyarrow`.

Data limitations found while testing on real data: the flow table's time column was read as text while the price time was a real date, so the join was forced to a common type; the flow file mixed sub-second precisions, so the parser now accepts either; and one coin (XRP) never downloaded, so it is absent from the current sample until the downloader is re-run. The last two would have broken the full 433-coin build and were caught early on the 9-coin sample.

### The split

Hold back the final ~1 year as an untouched test set, train on all earlier history, and drop a one-horizon embargo at the seam so no training row can peek into the test period. A full rolling walk-forward is the planned upgrade (`tasks/data-standards.md`). The split is executed just before training (the Model Training section below), and variable selection uses the training window only, so the hold-out stays blind.

### Post-split audit (representativeness and imbalance)

This is not stratification. We split by time on purpose, so we cannot force class proportions; instead we audit whether the temporal, multi-coin split is representative, on three fronts, in order of importance for this data.

1. Panel and coin composition. The pool mixes hundreds of coins with very unequal history (BTC to 2017, newer coins about two years), so the model is implicitly weighted toward long-history coins and the regimes they lived through. Check the row share per coin in train and test, the per-coin base rate on each side, and which coins appear in only one window (late listings, delistings). This coin dimension, not response-class balance, is what bites first.

2. Temporal drift. Because the split is by time, the hold-out can be a different regime, so a NO-GO with large drift means regime change, not absence of edge. Compare the label rate and each feature's distribution between the training years and the test year: base-rate shift (two-proportion z-test), continuous-feature drift (Kolmogorov-Smirnov plus a Population Stability Index, ranked, flag PSI above 0.10 and 0.25), and proportion shift on the binary and categorical features (the `f_tl_cdl_*` candlestick family, any regime flags). Re-assert that the embargo separates the last training label from the first test bar, and print both date spans.

3. Label imbalance. The barrier label is roughly 0.32 positive. We act on calibrated probabilities above a 0.60 confidence threshold rather than predicting by argmax, and `class_weight="balanced"` distorts exactly those probabilities, so class-weighting (and SMOTE, fit strictly within the training fold) are candidates to test against the natural distribution, graded on the after-fee metric and on calibration, not defaults. Any resampling stays inside the training window; cross-validation is embargoed and time-indexed, never shuffled KFold.

Cohen's Kappa, per-class precision, recall and F1, the confusion matrix and OOB are diagnostics that show whether the minority class is learned and whether a treatment helps. They do not decide the outcome; the after-fee Metric 2 (net expectancy per confident trade, against buy-and-hold and a coin-flip) remains the deciding number.

### What runs next

The cell below surfaces the active knobs (split size, trade filter, costs, label), read-only from the scripts. Then Import Data loads the prepared dataset, Feature Variables lists the candidate families, and the demonstration cell shows the cleaning and the screen tally on a coin sample. Modelling (Variable Selection onward) follows.

In [2]:
# Knobs live in the scripts so the notebook cannot drift. Read-only here.
CONFIG = dict(
    oos_days     = t1.OOS_DAYS,         # final year held out of sample
    embargo_days = t1.EMBARGO_DAYS,     # = label horizon in days
    conf_hi      = tm.CONF_HI,          # confidence filter act-long (Keller Metric 1)
    conf_lo      = tm.CONF_LO,          # act-short / stand-aside
    cost_pct     = tm.COST_PCT,         # round-trip drag (fee + slippage), Metric 2
)
lab = bd.LABEL
print(f"label: +{lab['tgt_atr']} ATR before -{lab['stp_atr']} ATR within {lab['horizon_bars']} bars "
      f"({lab['horizon_bars']//bd.BARS_PER_DAY}d)  .  hold out final {CONFIG['oos_days']}d, "
      f"embargo +/-{CONFIG['embargo_days']}d  .  confidence {CONFIG['conf_lo']}-{CONFIG['conf_hi']}  .  "
      f"cost {CONFIG['cost_pct']:.2f}% round trip")

label: +2.0 ATR before -1.0 ATR within 48 bars (2d)  .  hold out final 365d, embargo +/-2d  .  confidence 0.4-0.6  .  cost 0.20% round trip


### Import Data

Loads the prepared 1h dataset at `inputs/binance-data/dataset_1h_allmarket.parquet` (via `bd.DATASET_PATH`), built offline from the `data.binance.vision` archives by `inputs/build_dataset_1h.py`; `t1.load` reads it and keeps only the point-in-time `in_sample` rows. If it has not been built, the cell prints the build command. The cleaning, screen and label that produced this file are described once in Data Preparation above, so they are not repeated here.

The build is incremental and runs on demand via `tasks/run_auto_eval.sh`; it can also run nightly from cron, pulling the latest 24 hours, rebuilding the dataset, and retraining, so the load always reflects the most recent full day:

```bash
# 02:00 daily: refresh data -> rebuild dataset -> retrain
0 2 * * *  cd /Volumes/PortableSSD/Github/day-trader && \
  .venv/bin/python inputs/binance-data/flow_data.py --interval 1h --all-market && \
  .venv/bin/python inputs/build_dataset_1h.py && \
  .venv/bin/python inputs/train_model_1h.py
```

The cell below is reference-only: it reads the live build and reports when each file was produced, so the provenance is always visible. The trade-flow feature is `flow_imbalance = 2 * (taker_buy_base / volume) - 1`, the signed share of each bar's volume lifting the ask, in [-1, +1].

In [3]:
# Import Data: build provenance (how/when the data was built + the build config) AND the load +
# characteristics, in one cell. Reads the LIVE config from build_dataset_1h and the file
# timestamps; does NOT rebuild anything.
import os, datetime as _dt

def _built(p):
    base = os.path.splitext(p)[0]
    for ext in (".parquet", ".csv"):
        f = base + ext
        if os.path.exists(f):
            return (f"{os.path.basename(f)}  "
                    f"{_dt.datetime.fromtimestamp(os.path.getmtime(f)):%Y-%m-%d %H:%M}, "
                    f"{os.path.getsize(f)/1e6:.0f} MB")
    return "NOT BUILT YET"

# --- provenance + build configuration (nothing loaded yet) ---
print("SOURCE / LOCATION   (Binance spot, 1-hour bars, data.binance.vision archives)")
print(f"  klines root : {bd.DEFAULT_KLINES_ROOT}")
print(f"  flow table  : {_built(bd.DEFAULT_FLOW)}")
print(f"  dataset     : {_built(bd.DATASET_PATH)}")
print("  storage     : Parquet (columnar, compressed, dtype-preserving); CSV is the fallback")
print("\nTIME WINDOWS  (counts of bars; 1 bar = 1 hour)")
print(f"  wall-clock (daily x24): {bd.WC}")
print(f"  intraday              : {bd.HR}")
print(f"\nLABEL  (ATR-scaled triple barrier): {bd.LABEL}")
print(f"SCREEN gates                       : {bd.SCREEN}")
print(f"DATA-QUALITY gate                  : {bd.DATA_QUALITY}")
print("\nTo (re)build offline from the .venv (reference - not run here):")
print("  .venv/bin/python inputs/binance-data/flow_data.py --interval 1h --all-market")
print("  .venv/bin/python inputs/build_dataset_1h.py")

# --- load the prepared dataset + report its characteristics ---
# bd.read_frame prefers the Parquet file (real dtypes kept, ~2.6x smaller, loads in seconds);
# it falls back to the CSV if only that exists.
raw = bd.read_frame(bd.DATASET_PATH)
if raw is not None:
    raw = raw.sort_values("datetime").reset_index(drop=True)
    feat = bd.feature_columns(raw)
    insamp = raw["in_sample"] if "in_sample" in raw.columns else pd.Series(True, index=raw.index)
    df = raw[insamp].reset_index(drop=True)        # model on the point-in-time in-sample rows
    print(f"\nLOADED {len(raw):,} rows from Parquet")
    print(f"  in-sample rows     : {len(df):,}  ({len(df)/len(raw):.1%} of total)")
    print(f"  coins              : {raw['symbol'].nunique()}")
    print(f"  features           : {len(feat)}")
    print(f"  date range         : {raw['datetime'].min()}  ->  {raw['datetime'].max()}")
    print(f"  base rate all / in-sample : {raw['label'].mean():.3f} / {df['label'].mean():.3f}")
    fams = [("wall-clock", "f_wc_"), ("intraday", "f_hr_"), ("in-house TA", "f_ta_"),
            ("pandas-ta", "f_ta_pta_"), ("TA-Lib", "f_tl_"), ("flow", "f_flow_")]
    parts = []
    for nm, pre in fams:
        if pre == "f_ta_":
            cols = [c for c in feat if c.startswith("f_ta_") and not c.startswith("f_ta_pta_")]
        else:
            cols = [c for c in feat if c.startswith(pre)]
        parts.append(f"{nm} {len(cols)}")
    print("  feature families   : " + ", ".join(parts))
    print(f"  in-sample rows with any NaN feature: {int(df[feat].isna().any(axis=1).sum()):,}")
    print("  rows per coin:")
    show_table(raw["symbol"].value_counts().rename("rows").to_frame())
else:
    df = None; feat = []
    print("\n1h dataset not built yet. Build it from the project .venv:\n"
          "  .venv/bin/python inputs/build_dataset_1h.py\n"
          "(or a coin subset:  .venv/bin/python inputs/build_dataset_1h.py -s BTCUSDT ETHUSDT ...)")

SOURCE / LOCATION   (Binance spot, 1-hour bars, data.binance.vision archives)
  klines root : /Volumes/PortableSSD/Github/day-trader/inputs/binance-data/klines_1h
  flow table  : flow_1h.parquet  2026-06-21 10:36, 458 MB
  dataset     : dataset_1h_allmarket.parquet  2026-06-21 11:15, 684 MB
  storage     : Parquet (columnar, compressed, dtype-preserving); CSV is the fallback

TIME WINDOWS  (counts of bars; 1 bar = 1 hour)
  wall-clock (daily x24): {'ema_fast': 336, 'ema_mid': 2184, 'ema_slow': 3000, 'rsi': 336, 'bb': 336, 'bb_std': 2.0, 'atr': 336, 'rv_short': 168, 'rv_long': 720, 'vol': 480, 'mom': [120, 240, 480, 1440]}
  intraday              : {'ema_fast': 12, 'ema_mid': 26, 'ema_slow': 50, 'rsi': 14, 'bb': 20, 'bb_std': 2.0, 'atr': 14, 'rv_short': 24, 'rv_long': 168, 'vol': 24, 'mom': [6, 12, 24, 72, 168]}

LABEL  (ATR-scaled triple barrier): {'tgt_atr': 2.0, 'stp_atr': 1.0, 'horizon_bars': 48, 'atr_len': 14}
SCREEN gates                       : {'min_quote_volume_usdt': 30000000,


LOADED 1,640,660 rows from Parquet
  in-sample rows     : 458,539  (27.9% of total)
  coins              : 47
  features           : 61
  date range         : 2017-10-16 11:00:00  ->  2026-06-18 23:00:00
  base rate all / in-sample : 0.322 / 0.313
  feature families   : wall-clock 13, intraday 14, in-house TA 8, pandas-ta 7, TA-Lib 15, flow 4
  in-sample rows with any NaN feature: 0
  rows per coin:


,rows
symbol,
BTC/USDT,75229
ETH/USDT,75229
BNB/USDT,73293
LTC/USDT,72405
ADA/USDT,69437
LINK/USDT,62715
FET/USDT,61837
DASH/USDT,61177
DOGE/USDT,58971


### Feature Variables

The candidate set is broad by design. The variable-selection that follows, including the likelihood-ratio 
tests and elastic-net regularization paths, are designed to prune it. It spans two window families (a wall-clock family = the daily windows x24, and a shorter intraday family), an in-house extra-indicator block
(Williams %R, Stochastic, CCI, CMF, MFI, ADX/DMI, Aroon), the trade-flow imbalance, and - when run
from the `.venv` - optional pandas-ta (PPO, TRIX, Vortex, CMO, Fisher, Chande Kroll Stop) and
TA-Lib (SAR, MAMA, Ultimate Oscillator, Hilbert cycle features, candlestick patterns) layers. All
causal and scale-invariant.

**Variables in the code cell below** - this cell just counts and summarises the predictors; nothing new is built:

| variable | what it is, in plain terms | type / where it comes from | what it does here |
| --- | --- | --- | --- |
| `feat` | the names of every predictor column the model may use | `list[str]`, from the Import Data cell (`bd.feature_columns`) | the full candidate set being summarised |
| `df` | the in-sample dataset (the rows kept as tradable) | `DataFrame`, from the Import Data cell | source of the summary statistics |
| `fam` | the feature families paired with their column-name prefixes (e.g. wall-clock -> `f_wc_`) | `list` of (name, prefix) | groups the predictors so each family can be counted |
| `pre` | one family's column-name prefix | `str` (loop variable) | picks out that family's columns |
| `cols` | the predictor columns that belong to one family | `list[str]` | counted, to show how many features each family contributes |

In [4]:
if df is not None:
    fam = [("wall-clock (wc)", "f_wc_"), ("intraday (hr)", "f_hr_"),
           ("in-house TA", "f_ta_"), ("pandas-ta", "f_ta_pta_"),
           ("TA-Lib", "f_tl_"), ("flow", "f_flow_")]
    for name, pre in fam:
        if pre == "f_ta_":
            cols = [c for c in feat if c.startswith("f_ta_") and not c.startswith("f_ta_pta_")]
        else:
            cols = [c for c in feat if c.startswith(pre)]
        print(f"  {name:16s}: {len(cols)}")
    print(f"  {'TOTAL':16s}: {len(feat)}")
    show_table(df[feat].describe().T[["mean", "std", "min", "max"]].round(3))

  wall-clock (wc) : 13
  intraday (hr)   : 14
  in-house TA     : 8
  pandas-ta       : 7
  TA-Lib          : 15
  flow            : 4
  TOTAL           : 61


,mean,std,min,max
f_wc_ema_fast_mid,-0.018,0.194,-1.729,0.655
f_wc_ema_mid_slow,-0.006,0.064,-0.566,0.153
f_wc_rsi,0.502,0.026,0.372,0.647
f_wc_bb_pos,0.531,0.358,-1.825,2.594
f_wc_atr_pct,0.014,0.006,0.002,0.076
f_wc_rv_short,0.009,0.004,0.002,0.049
f_wc_rv_long,0.010,0.004,0.002,0.055
f_wc_rv_ratio,0.957,0.247,0.257,2.037
f_wc_vol_ratio,1.032,1.123,0.001,129.313
f_wc_mom_120,0.011,0.097,-0.716,0.801


### Data Processing (demonstration)

The cleaning and screening are defined in Data Preparation above and run inside `build_dataset_1h.py`. The cell below demonstrates them on a coin sample so the logic is visible and auditable: it shows the timestamp-unit handling, the per-coin screen tally (rows kept versus dropped), and the gap-gate pass or fail.

| variable | what it is, in plain terms | type / where it comes from | what it shows |
| --- | --- | --- | --- |
| `mixed_units` | two example timestamps in different units, one in milliseconds, one in microseconds | a short list of numbers (made up here) | the loader reads either unit correctly **(a)** |
| `mixed_prec` | two example time strings, one with fractional seconds, one without | a short list of text (made up here) | the parser accepts both precisions **(b)** |
| `raw` | the whole loaded dataset, with the keep/drop (`in_sample`) flag still attached | a table, from the Import Data cell | the starting point for the screen tally **(c)** |
| `eff` | per coin: how many rows exist versus how many were kept as tradable | a table, grouped by coin | how much the screen and warmup trimmed **(c)** |
| `d` | one coin's raw hourly price history | a table, loaded by `bd.load_coin` | the input to the gap check **(d)** |
| `q` | one coin's completeness stats: bars present, bars missing, biggest gap | a small summary (`dict`) | checked against the quality limits **(d)** |
| `rows` | the gap-check result for each sampled coin, with a pass/fail | a list turned into a table | which coins are clean enough to keep **(d)** |

In [5]:
# Data Processing - demonstration on the real data (auditable; this does NOT rebuild the dataset).
import pandas as pd

# (a) Timestamp normalisation: a millisecond stamp (~1e12) and a microsecond stamp (~1e15) both
#     parse to the correct datetime. bd._to_datetime classifies each value by magnitude.
mixed_units = pd.Series([1521212400000, 1748358000000000])     # one ms, one microsecond
print("(a) timestamp normalisation  ->", [str(t) for t in bd._to_datetime(mixed_units)])

# (b) Mixed sub-second precision (the flow_1h.csv bug): one value has microseconds, one does not.
#     A fixed format crashes; format='mixed' parses both. This is the fix in flow_block().
mixed_prec = pd.Series(["2018-03-16 15:00:00", "2018-03-16 16:00:00.000000"])
print("(b) mixed-precision parse    ->", [str(t) for t in pd.to_datetime(mixed_prec, format="mixed")])

# (c) Point-in-time screen + warmup effect, per coin (uses the dataset loaded in Import Data).
print("\n(c) point-in-time screen + warmup (rows kept vs in-sample), per coin:")
if "raw" in globals() and raw is not None:
    eff = (raw.groupby("symbol")
              .agg(rows=("label", "size"), in_sample=("in_sample", "sum"))
              .assign(in_sample_pct=lambda d: (d["in_sample"] / d["rows"] * 100).round(1))
              .sort_values("rows", ascending=False))
    show_table(eff)
else:
    print("    run the Import Data cell first (it defines `raw`).")

# (d) Gap gate: hourly-bar completeness of a few coins vs the DATA_QUALITY thresholds. A coin is
#     excluded if too many bars are missing or one gap is too long.
print(f"\n(d) gap gate   thresholds = {bd.DATA_QUALITY}")
rows = []
for sym in ["BTCUSDT", "ETHUSDT", "AVAXUSDT"]:
    d = bd.load_coin(bd.DEFAULT_KLINES_ROOT, sym)
    if len(d):
        q = bd.gap_stats(d)
        rows.append({"coin": sym, "bars": q["actual"], "gap_ratio": round(q["gap_ratio"], 4),
                     "max_gap_hours": round(q["max_gap_hours"], 1), "passes": bd.passes_quality(q)})
show_table(pd.DataFrame(rows))

(a) timestamp normalisation  -> ['2018-03-16 15:00:00', '2025-05-27 15:00:00']
(b) mixed-precision parse    -> ['2018-03-16 15:00:00', '2018-03-16 16:00:00']

(c) point-in-time screen + warmup (rows kept vs in-sample), per coin:


,rows,in_sample,in_sample_pct
symbol,,,
ETH/USDT,75229,59543,79.1
BTC/USDT,75229,54154,72.0
BNB/USDT,73293,47723,65.1
LTC/USDT,72405,32527,44.9
ADA/USDT,69437,33338,48.0
LINK/USDT,62715,29987,47.8
FET/USDT,61837,5869,9.5
DASH/USDT,61177,1906,3.1
DOGE/USDT,58971,35421,60.1



(d) gap gate   thresholds = {'max_gap_ratio': 0.02, 'max_single_gap_hours': 72}


,coin,bars,gap_ratio,max_gap_hours,passes
0,BTCUSDT,77389,0.0016,33.5,True
1,ETHUSDT,77389,0.0016,33.5,True
2,AVAXUSDT,50327,0.0004,5.0,True


## Variable Selection

Stage ii: prune the broad candidate set to the subset that earns its place, BEFORE the head-to-head
and before any hyperparameter tuning, using the TRAINING window only so the final-year hold-out
stays untouched (otherwise the blind score is no longer blind).

- **Elastic-glmnet path.** A regularized logistic regression (`saga`, `penalty="elasticnet"`) over 
  the standardized features on the training split; the features with non-zero coefficients at the 
  chosen penalty are the selected subset. Sweep `l1_ratio` and `C` (the lambda / mixing knobs) to trace the path.
- **Likelihood-ratio test.** For the logistic model, compare nested fits (with vs without a feature
  or block) by the chi-square of the deviance difference. The tree models (RF, LightGBM) lack that
  nested-likelihood structure, so use permutation importance for them instead.

The selected subset feeds the models below. A Brier score derived as RMSE of outcome probabilities 
is printed as a calibration check.

In [6]:
# Stage ii: elastic-net variable selection on the TRAINING split only (the blind year is untouched).
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss

if df is not None and feat:
    tr_vs, te_vs, _cut = t1.split(df)                 # same final-year split; select on TRAIN
    Xtr_vs = tr_vs[feat].astype(float).fillna(0.0); ytr_vs = tr_vs["label"]
    sc = StandardScaler().fit(Xtr_vs)
    enet = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, C=0.1,
                              max_iter=5000, class_weight="balanced")
    enet.fit(sc.transform(Xtr_vs), ytr_vs)
    coef = pd.Series(enet.coef_[0], index=feat)
    SELECTED_FEATURES = list(coef[coef.abs() > 1e-6].abs().sort_values(ascending=False).index)
    print(f"elastic-net kept {len(SELECTED_FEATURES)} of {len(feat)} features (l1_ratio=0.5, C=0.1).")
    print("top kept:", ", ".join(SELECTED_FEATURES[:15]))
    p_tr = enet.predict_proba(sc.transform(Xtr_vs))[:, 1]
    print(f"in-sample Brier (lower = better calibrated): {brier_score_loss(ytr_vs, p_tr):.4f}")
    print("\nTo train the head-to-head on this subset, set  feat = SELECTED_FEATURES  then re-run the "
          "training cells. Sweep l1_ratio / C for the full path; add a per-block LRT as needed.")
else:
    SELECTED_FEATURES = []
    print("build the 1h dataset first (see Import Data).")

elastic-net kept 61 of 61 features (l1_ratio=0.5, C=0.1).
top kept: f_hr_mom_24, f_hr_ema_fast_mid, f_ta_cci, f_ta_aroon_osc, f_hr_rsi, f_ta_pta_cmo, f_hr_bb_pos, f_ta_pta_cksp_long_dist, f_wc_atr_pct, f_hr_mom_12, f_ta_pta_ppo_hist, f_wc_ema_fast_mid, f_wc_rv_long, f_wc_bb_pos, f_wc_mom_240
in-sample Brier (lower = better calibrated): 0.2483

To train the head-to-head on this subset, set  feat = SELECTED_FEATURES  then re-run the training cells. Sweep l1_ratio / C for the full path; add a per-block LRT as needed.


## Training Regime

The split is executed here (in the cell just below), then the models are trained on all history before the final-year cut and scored once on the held-out year; the representativeness and imbalance audit for this split is described in Data Preparation above. Three models compete: logistic regression, random forest, and LightGBM (Tier 1). Each is reported at the 0.5 threshold and under the 60/40 confidence filter (Keller Metric 1); read precision against the base rate, which sits well below 0.5. The full record - Metric 1, Metric 2 (P&L after the 0.20% cost), and Metric 3 (AUC by volatility regime) - is written to `outputs/AA-evals/` by `eval_report.write_comparison`, tagged "head-to-head (1h)", so it sits beside the other runs in `evaluation-scores.md`.

In [7]:
# Split and scoring come from the scripts: t1.split for the 1h final-year hold-out; tm.evaluate,
# tm.build_models, tm.confidence_filtered shared with inputs/train_model.py. Nothing reimplemented.
train, test, cut = t1.split(df)
Xtr, ytr, Xte, yte = train[feat], train["label"], test[feat], test["label"]
base_te = yte.mean()
span = lambda d: f"{d['datetime'].min().date()} to {d['datetime'].max().date()}"
print(f"train {len(train):,} ({span(train)})  .  test {len(test):,} ({span(test)})  .  "
      f"cut {pd.Timestamp(cut).date()}  embargo +/-{t1.EMBARGO_DAYS}d  .  base rate {base_te:.3f}")

train 381,207 (2017-12-15 to 2025-06-16)  .  test 76,789 (2025-06-19 to 2026-06-18)  .  cut 2025-06-18  embargo +/-2d  .  base rate 0.303


In [8]:
# Fast interactive pass. The full-fidelity 3-model run on ALL history is inputs/train_model_1h.py
# (script / auto-eval). The heavy part here is evaluate()'s 5-fold TimeSeriesSplit CV on each model
# (3 models x (5 CV fits + 1 final fit) on ~381k rows = minutes). For the notebook we fit a bounded
# sample taken across the whole period and re-sorted by time, so the time-series CV stays
# chronological. Raise TRAIN_CAP or set it to None for the full run (slow).
TRAIN_CAP = 80_000
if TRAIN_CAP is None or len(train) <= TRAIN_CAP:
    tr_fit = train
else:
    tr_fit = train.sample(TRAIN_CAP, random_state=0).sort_values("datetime")
Xtr_f, ytr_f = tr_fit[feat], tr_fit["label"]
print(f"fitting on {len(tr_fit):,} of {len(train):,} train rows "
      f"({tr_fit['datetime'].min().date()} to {tr_fit['datetime'].max().date()})"
      + ("  [sampled for the notebook; full run is inputs/train_model_1h.py]"
         if (TRAIN_CAP and len(train) > TRAIN_CAP) else ""))

lines, scored = [], []
for name, mdl in tm.build_models(HAVE_LGBM):
    prec, m = tm.evaluate(name, mdl, Xtr_f, ytr_f, Xte, yte, base_te, lines)
    scored.append((prec, name, mdl, m))
print("\n".join(lines))
_, best_name, best_model, best = max(scored, key=lambda t: t[0])
print(f"\nchosen: {best_name}  (precision {best['prec']:.3f} vs base {base_te:.3f}, AUC {best['auc']:.3f})")

fitting on 80,000 of 381,207 train rows (2017-12-16 to 2025-06-16)  [sampled for the notebook; full run is inputs/train_model_1h.py]



--- LogisticRegression ---
  train CV ROC-AUC (5-fold TimeSeriesSplit): 0.528
  test accuracy : 0.484
  test precision(buy): 0.307   (base rate 0.303)
  test recall(buy)   : 0.560
  test ROC-AUC       : 0.504
  confusion matrix [rows=true 0/1, cols=pred 0/1]:
   [[24164, 29352], [10241, 13032]]
  precision lift over base rate: +0.004
  confidence filter (act if p>=0.60 or p<=0.40): keeps 2.7% of test rows (n=2075)
    precision(buy) 0.285  recall 0.663  F1 0.398  (base rate 0.303)

--- RandomForest ---
  train CV ROC-AUC (5-fold TimeSeriesSplit): 0.526
  test accuracy : 0.489
  test precision(buy): 0.310   (base rate 0.303)
  test recall(buy)   : 0.557
  test ROC-AUC       : 0.509
  confusion matrix [rows=true 0/1, cols=pred 0/1]:
   [[24627, 28889], [10319, 12954]]
  precision lift over base rate: +0.007
  confidence filter (act if p>=0.60 or p<=0.40): keeps 1.2% of test rows (n=931)
    precision(buy) 0.000  recall 0.000  F1 0.000  (base rate 0.303)

--- LightGBM ---
  train CV ROC-

In [9]:
# Honesty gate: precision must clearly beat the base rate and AUC clear 0.55. It does not
# trade; Metric 2 (next cell) is the money test.
margin = best["prec"] - base_te
go = (best["prec"] > base_te + 0.05) and (best["auc"] > 0.55)
verdict = "GO" if go else "NO-GO"
print("HONESTY GATE:", "GO (edge survives out-of-sample)" if go
      else "NO-GO (no demonstrable edge - do not trade)")
joblib.dump({"model": best_model, "features": feat, "name": best_name,
             "trained_through": str(pd.Timestamp(cut).date()),
             "test_base_rate": float(base_te), "go": bool(go)}, MODEL_DIR / "model_1h.joblib")
summary = (f"{best_name}: test precision(buy)={best['prec']:.3f} base_rate={base_te:.3f} "
           f"lift={margin:+.3f} AUC={best['auc']:.3f} recall={best['rec']:.3f} acc={best['acc']:.3f} -> {verdict}")
(MODEL_DIR / "model_metrics_1h.txt").write_text(summary + "\n\n" + "\n".join(lines) + "\n")
print("saved", MODEL_DIR / "model_1h.joblib", "and model_metrics_1h.txt")

HONESTY GATE: NO-GO (no demonstrable edge - do not trade)
saved outputs/3B-model-training/model_1h.joblib and model_metrics_1h.txt


In [10]:
# Metrics 2 (P&L after costs) and 3 (regime-stratified AUC) plus AA-evals bookkeeping, from
# inputs/eval_report.py - the same record the script writes. Appends one row to
# outputs/AA-evals/evaluation-scores.md (+ .pdf, .docx), tagged "head-to-head (1h)".
imp = getattr(best_model, "feature_importances_", None)
regime = "f_wc_rv_long" if "f_wc_rv_long" in test.columns else next((c for c in feat if "rv_long" in c), None)
meta = dict(dataset_rows=len(df), n_features=len(feat),
            train_rows=len(train), test_rows=len(test), base_rate=float(base_te),
            cut=str(pd.Timestamp(cut).date()), embargo=t1.EMBARGO_DAYS,
            conf_hi=tm.CONF_HI, conf_lo=tm.CONF_LO, chosen=best_name, verdict=verdict,
            eval_type="head-to-head (1h)",
            dataset_label=f"{len(df):,}r / {len(feat)}f (1h all-market)",
            fi_names=list(feat) if imp is not None else None,
            fi_values=[float(v) for v in imp] if imp is not None else None,
            regime_vol=test[regime].tolist() if regime else None,
            trade_ret=test["trade_ret"].tolist() if "trade_ret" in test.columns else None,
            cost_pct=tm.COST_PCT)
results = [m for (_, _, _, m) in scored]
rec = eval_report.write_comparison(str(OUTPUTS / "AA-evals"), results, yte, meta)
print("evaluation record:", rec["md"])

(reportlab not installed: skipped evaluation-scores.pdf. pip install reportlab)
(python-docx not installed: skipped evaluation-scores.docx. pip install python-docx)
evaluation record: outputs/AA-evals/2026-06-21/eval-head-to-head-20260621.md


In [11]:
# Teaching display: the strongest tree model's top features. The AA-evals record above
# already saves this chart; this is just an inline look.
tree = next((mdl for (_, name, mdl, _) in scored
             if name in ("LightGBM", "RandomForest")), None)
imp = getattr(tree, "feature_importances_", None) if tree is not None else None
if imp is not None:
    order = np.argsort(imp)[::-1][:15]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh([feat[i] for i in order][::-1], imp[order][::-1], color="#0B3D66")
    ax.set_title(f"{tree.__class__.__name__}: top 15 feature importances")
    fig.tight_layout()
    (OUTPUTS / "PNG").mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUTS / "PNG" / "3B-feature-importance.png", dpi=160)
    plt.show()

## Model Tuning

Two sweeps, both scored on the after-fee out-of-sample scoreboard and logged to the consolidated
`outputs/AA-evals/evaluation-scores.md`:

- **Priority 1b - label geometry** (`inputs/sweep_label_1h.py`): the ATR-scaled triple barrier
  swept over (target, stop, horizon). Each cell reports its own base rate, the breakeven win rate
  stop/(stop+target), and net P&L/trade after cost, so a lopsided geometry is obvious.
- **Priority 1a - exit geometry** (`inputs/walkforward.py`): stop and take-profit, plus per-coin
  trailing stops and a time-decaying take-profit - the same question from the exit side.

Settle the two together, then unify the forked label/exit configs. The broad feature set is left
broad here on purpose; the variable-selection pass (likelihood-ratio tests, elastic-net paths)
prunes predictors before final fitting.

In [12]:
# The sweeps are full-market and run as scripts from the project .venv:
#   .venv/bin/python inputs/sweep_label_1h.py    # Priority 1b: label geometry (target, stop, horizon)
#   .venv/bin/python inputs/walkforward.py       # Priority 1a: exit geometry (stop, take-profit, trail)
# Both append to the consolidated scoreboard; here we render that one table so tuning reads off it.
from IPython.display import Markdown, display
scoreboard = OUTPUTS / "AA-evals" / "evaluation-scores.md"
if scoreboard.exists():
    display(Markdown(scoreboard.read_text()))
else:
    print("no evaluation-scores.md yet - run a model (cells above) or a sweep script first.")
# Quick inline 1b probe (small grid; the full grid is the script above):
#   import sweep_label_1h as sw
#   coins, scols = sw.precompute(bd.DEFAULT_KLINES_ROOT, bd.DEFAULT_FLOW_CSV, None)
#   sw.score_cell(coins, scols, 3.0, 1.0, 48, tm.COST_PCT/100.0, tm.CONF_HI)

# Evaluation scores

One row per evaluation run, newest first. Each row links to its full record.

## What counts as a "buy" (the label)

The model predicts one yes/no event, not a price or a return size.

- A "buy" is set by a **triple-barrier** test (Lopez de Prado). From each day's close, draw three lines: an upper barrier at **+10%**, a lower barrier at **-5%**, and a time barrier **20 days** out.
- Walk forward day by day. If price reaches **+10% before** it falls to -5%, that day is a buy (**label = 1**).
- If it hits **-5% first**, or 20 days pass without reaching +10%, it is a **0**.
- On a day where both could have happened, we assume the **stop (-5%) hit first**, so the label never flatters itself.
- So the question the model answers is: will a +10% move arrive before a -5% drawdown within 20 days?

## How precision is scored

Every test day has two facts: what the model said (buy or not, at the 0.5 cut) and what actually happened. That gives four outcomes:

- **True positive** - said buy, and it was a buy. A good call.
- **False positive** - said buy, but it was not. A bad trade that spends real money.
- **False negative** - said no, but it was a buy. A missed chance.
- **True negative** - said no, and it was not. Correctly stood aside.

- **Precision = true positives / (true positives + false positives).** In plain words: every time the model shouts "buy", how often is it right?
- Precision only punishes bad trades (false positives) and ignores missed chances (false negatives). That is deliberate: a bad trade loses money now, a missed chance only costs an opportunity that comes around again.
- Recall is the mirror (of all the real buys, how many we caught). We rank models by precision, not recall, because being right when we act matters more than acting often.

## What the columns mean

- **date** - the day the evaluation was run.
- **evaluation type** - which kind of evaluation. *Head-to-head*: several models trained on the same train/test split and compared. Later types: walk-forward backtest, tuning sweep, stability check.
- **dataset** - rows and feature count used (e.g. 26,762r / 32f).
- **best model** - the model kept, chosen by the highest Best Model Precision.
- **test AUC** - area under the ROC curve on the out-of-sample test set: the chance the model scores a real buy above a non-buy. 0.50 = no skill (a coin flip), 1.00 = perfect ranking; it does not depend on a threshold.
- **Best Model Precision** - the precision of the chosen model: of the days it called a buy, the share that were genuine buys.
- **Always Buys Precision** - the precision a mindless model that calls every day a buy would score, i.e. the share of all test days that were buys. The baseline to beat.
- **Precision Change (%)** - how much better the best model is than always-buying: (Best Model Precision / Always Buys Precision - 1) x 100. +0% = no better than mindless; above 0 = adding value. Verify it from the two columns to its left.
- **Net P&L/trade** - Keller Metric 2 in one number: the average return of a trade the model takes (probability >= 0.60), after the 0.20% round-trip cost (Binance.com spot with BNB, plus slippage). Above 0 = a model-picked trade makes money after fees. This is the number to maximise when tuning.
- **trades** - how many trades the model would have taken in the test window (probability >= 0.60). Too few trades and the P&L is noise.
- **verdict** - GO only if Best Model Precision clearly beats Always Buys Precision and AUC clears 0.55; otherwise NO-GO.
- **record** - links to the full per-run report (markdown for the numbers, HTML for the charts).

Each per-run record also reports Keller Metric 1 (precision and recall at the 60/40 trading threshold) and Metric 3 (AUC split by volatility regime).

## Runs

| date | evaluation type | dataset | best model | test AUC | Best Model Precision | Always Buys Precision | Precision Change (%) | Net P&L/trade | trades | verdict | record |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-06-21 | head-to-head (1h) | 458,539r / 61f (1h all-market) | LightGBM | 0.510 | 0.311 | 0.303 | +2.5% | -0.38% | 7,348 | NO-GO | [md](2026-06-21/eval-head-to-head-20260621.md) / [html](2026-06-21/eval-head-to-head-20260621.html) |
| 2026-06-21 | head-to-head (1h) | 458,539r / 61f (1h all-market) | LightGBM | 0.510 | 0.311 | 0.303 | +2.5% | -0.38% | 7,348 | NO-GO | [md](2026-06-21/eval-head-to-head-20260621.md) / [html](2026-06-21/eval-head-to-head-20260621.html) |
| 2026-06-21 | head-to-head (1h) | 458,539r / 61f (1h all-market) | LightGBM | 0.513 | 0.312 | 0.303 | +2.9% | -0.35% | 5,543 | NO-GO | [md](2026-06-21/eval-head-to-head-20260621.md) / [html](2026-06-21/eval-head-to-head-20260621.html) |
| 2026-06-21 | head-to-head | 26,772r / 32f | LightGBM | 0.528 | 0.326 | 0.305 | +7.0% | +0.20% | 1,398 | NO-GO | [md](2026-06-21/eval-head-to-head-20260621.md) / [html](2026-06-21/eval-head-to-head-20260621.html) |


## Model Assessment

A head-to-head scorecard in the style of a caret model-assessment table. For each model it reports
the tuned hyperparameters, the in-sample (Full) error, the cross-validated error, and their ratio.
Errors are RMSE and MAE on the predicted probabilities - here RMSE is the square root of the Brier
score, the caret-style classification RMSE. The RMSEratio (Full RMSE / CV RMSE) flags overfitting:
near 1 means the model generalises; well below 1 means it fits the training data far better than it
holds up out of sample. Cross-validation is time-ordered (not random folds), and the final year is
kept as a single blind test. The table and the best model are logged to the consolidated scoreboard
as a "tuning" row.

In [13]:
# Caret-style assessment (stage iv). Heavy on the full market, so it runs as a script from the .venv:
#   .venv/bin/python inputs/model_assessment_1h.py
# Here we render the latest assessment record if one exists.
import glob
import model_assessment_1h as ma
from IPython.display import Markdown, display
recs = sorted(glob.glob(str(OUTPUTS / "AA-evals" / "*" / "model-assessment-*.md")))
if recs:
    display(Markdown(open(recs[-1]).read()))
else:
    print("no assessment record yet - run:  .venv/bin/python inputs/model_assessment_1h.py")
# Quick inline run on two models (the full zoo + ensemble is the script above):
#   dfa = t1.load(bd.DATASET_PATH); fa = bd.feature_columns(dfa)
#   rows, tr, te, cut = ma.assess(dfa, fa, only=["logreg", "lightgbm"])

no assessment record yet - run:  .venv/bin/python inputs/model_assessment_1h.py


## Stability

Confirm any edge is not an artifact: parameter stability, results split by market type,
coin-flip and buy-and-hold baselines, an optional bootstrap on trade returns. Only then
paper trade, then a tiny live allocation. No live trading until a configuration clearly
beats buy-and-hold and a coin flip, out-of-sample and after fees.